In [1]:
# System
import os
import sys

os.environ["KERAS_BACKEND"] = "jax"
sys.path.append("../..")

In [2]:
# Setup
import json
from importlib import import_module

import numpy as np
from keras import ops
from rich.console import Console
from rich.table import Table

from src.models import (
    GradientBoostedDecisionTree,
    LearnableCutFlowParallel,
    LearnableCutFlowSequential,
    MultiLayerPerceptron,
)
from src.utils import load_model

In [3]:
# Parameters
# Dataset
dataset_name = "real1"  # *
n_samples = 200000
seed = 42
selected_feature_indices = [0, 2, 4, 5]  # *

# Model
centers = [80, 0.15, 0.025, 2, 2, 0.3]  # *
n_epochs = 200
batch_size = 512

# Plotting *
bins = 100
feature_names = ["mass", "c2_beta1", "c2_beta2", "d2_beta1", "d2_beta2", "tau21_beta1"]
feature_names_latex = [
    r"$M_{jet}$",
    r"$C_2^{\beta=1}$",
    r"$C_2^{\beta=2}$",
    r"$D_2^{\beta=1}$",
    r"$D_2^{\beta=2}$",
    r"$\tau_{21}^{\beta=1}$",
]

In [4]:
# Dataset *
load_data = import_module(f"src.datasets.{dataset_name}").load_data
(x_train, y_train), (x_test, y_test) = load_data(n_samples, seed)

x_train = x_train[:, selected_feature_indices]
x_test = x_test[:, selected_feature_indices]
centers = [centers[i] for i in selected_feature_indices]
feature_names = [feature_names[i] for i in selected_feature_indices]
feature_names_latex = [feature_names_latex[i] for i in selected_feature_indices]

selection = np.ones_like(x_train[:, 0], dtype=bool)
for i in range(x_train.shape[1]):
    p05 = np.percentile(x_train[:, i], 5)
    p95 = np.percentile(x_train[:, i], 95)
    selection = (p05 < x_train[:, i]) & (x_train[:, i] < p95) & selection

x_train = x_train[selection]
y_train = y_train[selection]

selection = np.ones_like(x_test[:, 0], dtype=bool)
for i in range(x_test.shape[1]):
    p05 = np.percentile(x_test[:, i], 5)
    p95 = np.percentile(x_test[:, i], 95)
    selection = (p05 < x_test[:, i]) & (x_test[:, i] < p95) & selection

x_test = x_test[selection]
y_test = y_test[selection]

print(f"{x_train.shape=}")
print(f"{y_train.shape=}")
print(f"{x_test.shape=}")
print(f"{y_test.shape=}")

x_train.shape=(76993, 4)
y_train.shape=(76993, 1)
x_test.shape=(76874, 4)
y_test.shape=(76874, 1)


In [5]:
# Model: gradient boosted decision tree
bdt = GradientBoostedDecisionTree(input_shape=x_train.shape, name="bdt")

bdt.compile(optimizer="adam", loss="crossentropy")
bdt.fit(x_train, y_train.squeeze(), epochs=n_epochs, batch_size=batch_size)

bdt.save(f"checkpoints/{bdt.name}.pkl")
ckpt_bdt = load_model(f"checkpoints/{bdt.name}.pkl")

In [6]:
# Model: multi-layer perceptron
mlp = MultiLayerPerceptron(input_shape=x_train.shape, name="mlp")

mlp.layers[1].adapt(x_train)
mlp.compile(optimizer="adam", loss="crossentropy")
mlp.fit(x_train, y_train, epochs=n_epochs, batch_size=batch_size)

mlp.save(f"checkpoints/{mlp.name}.keras")
ckpt_mlp = load_model(f"checkpoints/{mlp.name}.keras")

Epoch 1/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 8s 32ms/step - loss: 0.5143
Epoch 2/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3451
Epoch 3/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3258
Epoch 4/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3255
Epoch 5/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3255
Epoch 6/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3222
Epoch 7/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3268
Epoch 8/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3282
Epoch 9/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3205
Epoch 10/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3207
Epoch 11/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3211
Epoch 12/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3216
Epoch 13/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3239
Epoch 14/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3227
Epoch 15/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 

In [7]:
# Model: learnable cut flow parallel
lcf_par = LearnableCutFlowParallel(
    x_train.shape, centers, feature_names=feature_names, name="lcf_par"
)

lcf_par.normalization.adapt(x_train)
lcf_par.compile(optimizer="adam", loss="crossentropy")
lcf_par.fit(x_train, y_train, epochs=n_epochs, batch_size=batch_size)

lcf_par.save(f"checkpoints/{lcf_par.name}.keras")
ckpt_lcf_par = load_model(f"checkpoints/{lcf_par.name}.keras")

Epoch 1/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - loss: 0.3525
Epoch 2/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3397
Epoch 3/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3296
Epoch 4/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3216
Epoch 5/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3146
Epoch 6/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3102
Epoch 7/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3051
Epoch 8/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3026
Epoch 9/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2990
Epoch 10/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2970
Epoch 11/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2951
Epoch 12/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2936
Epoch 13/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2928
Epoch 14/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2911
Epoch 15/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 

In [8]:
# Model: learnable cut flow sequential
lcf_seq = LearnableCutFlowSequential(
    x_train.shape, centers, feature_names=feature_names, name="lcf_seq"
)

lcf_seq.normalization.adapt(x_train)
lcf_seq.compile(optimizer="adam", loss="crossentropy")
lcf_seq.fit(x_train, y_train, epochs=n_epochs, batch_size=batch_size)

lcf_seq.save(f"checkpoints/{lcf_seq.name}.keras")
ckpt_lcf_seq = load_model(f"checkpoints/{lcf_seq.name}.keras")

Epoch 1/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0962
Epoch 2/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0946
Epoch 3/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1169
Epoch 4/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1433
Epoch 5/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1524
Epoch 6/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1550
Epoch 7/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1545
Epoch 8/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1553
Epoch 9/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1551
Epoch 10/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1531
Epoch 11/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1561
Epoch 12/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1557
Epoch 13/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1569
Epoch 14/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1585
Epoch 15/200
151/151 ━━━━━━━━━━━━━━━━━━━━ 

In [9]:
# Analysis: metrics
results = {}
y_true = y_test

console = Console(force_jupyter=False)
table = Table(title="Model Performance Comparison")
table.add_column("#", justify="center", style="cyan", no_wrap=True)
table.add_column("Model", style="magenta")
table.add_column("TP", justify="right", style="green")
table.add_column("FP", justify="right", style="red")
table.add_column("Accuracy", justify="right", style="blue")
table.add_column("Precision", justify="right", style="blue")
table.add_column("Significance", justify="right", style="yellow")

for i, model in enumerate([ckpt_bdt, ckpt_mlp, ckpt_lcf_par, ckpt_lcf_seq]):
    y_pred = model.predict(x_test, batch_size=batch_size, verbose=0)
    y_pred = ops.all(y_pred > 0.5, axis=1, keepdims=True)

    tp = ops.sum((y_true == 1) & (y_pred == 1))
    fp = ops.sum((y_true == 0) & (y_pred == 1))
    tn = ops.sum((y_true == 0) & (y_pred == 0))
    fn = ops.sum((y_true == 1) & (y_pred == 0))

    n_preds_true = ops.add(tp, tn)
    n_preds_false = ops.add(fp, fn)
    n_positives = ops.add(tp, fp)
    n_samples = ops.add(n_preds_true, n_preds_false)

    accuracy = ops.divide(n_preds_true, n_samples)
    precision = ops.divide(tp, n_positives)
    significance = ops.divide(tp, ops.sqrt(fp))

    results[model.name] = {
        "tp": ops.convert_to_numpy(tp).tolist(),
        "fp": ops.convert_to_numpy(fp).tolist(),
        "accuracy": ops.convert_to_numpy(accuracy).tolist(),
        "precision": ops.convert_to_numpy(precision).tolist(),
        "significance": ops.convert_to_numpy(significance).tolist(),
    }

    table.add_row(
        str(i + 1),
        model.name,
        f"{tp:.0f}",
        f"{fp:.0f}",
        f"{accuracy:.4f}",
        f"{precision:.4f}",
        f"{significance:.4f}",
    )

console.print(table)

with open("results.json", "w") as f:
    json.dump(results, f, indent=4)

                    Model Performance Comparison                    
┏━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ # ┃ Model   ┃    TP ┃   FP ┃ Accuracy ┃ Precision ┃ Significance ┃
┡━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ 1 │ bdt     │ 41006 │ 6709 │   0.8657 │    0.8594 │     500.6319 │
│ 2 │ mlp     │ 40675 │ 6242 │   0.8674 │    0.8670 │     514.8322 │
│ 3 │ lcf_par │ 27964 │ 3710 │   0.7350 │    0.8829 │     459.1054 │
│ 4 │ lcf_seq │ 40573 │ 7187 │   0.8538 │    0.8495 │     478.5896 │
└───┴─────────┴───────┴──────┴──────────┴───────────┴──────────────┘
